# Seoul Ring Expressway — 3D label (spatial × temporal) regression demo

This notebook shows a spatio-temporal regression workflow where inputs X are (month, day-type, direction) and labels Y are a (segment × hour) grid.
- Features X: shape (B, D) where B is the number of (month, day-type, direction) groups; D is feature dim (e.g., 3).
- Labels Y: shape (B, S, T) where S is segment count, T is 24 hours.
- Model: `stnet` spatiotemporal model built with `Model` + `ModelConfig/PatchConfig`.
- Runtime: works on Colab T4 (CUDA 12.x) or CPU; adjusts config per device.
- Note: this is a shape/alignment demo; tweak hyperparameters for real training.


## 1. Environment and dependencies
Install the analysis stack (tensordict, h5py included). Triton ships with PyTorch, so we do not force-install it.


In [ ]:
# Colab/local common dependencies (assume stnet is installed separately)
import sys, importlib

def _ensure(pkg: str, mod: str | None = None) -> None:
    mod = mod or pkg
    try:
        importlib.import_module(mod)
    except Exception:
        import subprocess
        print(f"[install] {pkg} ...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', pkg], check=True)

# Core analytics stack (h5py/tensordict included; Triton ships with PyTorch)
_ensure('polars'); _ensure('tensordict'); _ensure('h5py')
_ensure('fastexcel'); _ensure('xlsxwriter'); _ensure('matplotlib'); _ensure('openpyxl')

import os, math
from typing import Any, Dict, List, Tuple, Optional
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.dpi'] = 120

# Detect Colab and mount Google Drive
try:
    import google.colab  # type: ignore
    IS_COLAB = True
except Exception:
    IS_COLAB = False

if IS_COLAB:
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive', force_remount=True)
        print('[info] Google Drive mounted at /content/drive')
    except Exception as e:
        print(f"[warn] Google Drive mount failed: {e!r}")

print(f'IS_COLAB = {IS_COLAB}')


In [ ]:
import torch, importlib, sys

def _ensure(pkg: str, mod: str | None = None) -> None:
    mod = mod or pkg
    try:
        importlib.import_module(mod)
    except Exception:
        import subprocess
        print(f"[install] {pkg} ...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', pkg], check=True)

# GPU-only packages: try installing only on CUDA 12.x (Colab T4 uses cu128)
if torch.cuda.is_available():
    cuda_ver = (torch.version.cuda or '').strip()
    if cuda_ver.startswith('12'):
        _ensure('transformer-engine-cu12', mod='transformer_engine')
        _ensure('transformer-engine-torch')
    else:
        print(f"[skip] transformer-engine (CUDA {cuda_ver})")
else:
    print('[skip] GPU-only extras (no CUDA visible)')


## 2. Input Excel path and output directory
Defaults to the bundled `raw_data.xlsx`. On Colab, if `/content/drive/My Drive/raw_data.xlsx` exists, it is used; otherwise we warn and fall back. Working directory is set to the file's parent.


In [ ]:
from typing import Union
import os
from pathlib import Path

ROOT = Path.cwd()
DEFAULT_RAW = ROOT / 'raw_data.xlsx'
EXCEL_PATH = DEFAULT_RAW

if IS_COLAB:
    drive_candidate = Path('/content/drive/My Drive/raw_data.xlsx')
    if drive_candidate.exists():
        EXCEL_PATH = drive_candidate
    else:
        print(f"[warn] Drive file missing: {drive_candidate}; using bundled raw_data.xlsx")

OUT_DIR = EXCEL_PATH.parent
os.makedirs(OUT_DIR, exist_ok=True)
os.chdir(OUT_DIR)

print('EXCEL_PATH:', EXCEL_PATH)
print('OUT_DIR   :', str(OUT_DIR))


## 3. Convert wide Excel to long format
Each sheet name encodes (month, weekday). We load every sheet, add route info, and unpivot hour columns into a long table (route, segment, direction, month, weekday, hour, value).


In [ ]:
# Each Excel sheet contains columns like '노선', '구간', '방향', '00시'~'23시'
import re
from openpyxl import load_workbook

def parse_sheet_name(name: str) -> Tuple[int, str]:
    """Extract (month, weekday) from sheet name. Example: '1월 월요일' -> (1, '월요일')"""

    # 1. Extract month
    m = re.search(r"(\d+)월", name)
    if not m:
        raise ValueError(f"Could not find month in sheet name: {name}")
    month = int(m.group(1))

    # 2. Extract weekday (avoid confusing the '월' character in the month substring)
    name_clean = name.replace(m.group(0), "")

    if "월" in name_clean: day_kind = "월요일"
    elif "화" in name_clean: day_kind = "화요일"
    elif "수" in name_clean: day_kind = "수요일"
    elif "목" in name_clean: day_kind = "목요일"
    elif "금" in name_clean: day_kind = "금요일"
    elif "토" in name_clean: day_kind = "토요일"
    elif "일" in name_clean: day_kind = "일요일"
    else:
        # Fallback for legacy values such as '평일' (weekday)
        raise ValueError(f"Could not find weekday in sheet name: {name}")

    return month, day_kind

HOURS = [f"{h:02d}시" for h in range(24)]

# Sheet list
workbook = load_workbook(EXCEL_PATH, read_only=True, data_only=True)
sheet_names: List[str] = list(workbook.sheetnames)
print(f"Total sheets: {len(sheet_names)}")
print("Sample sheets:", sheet_names[:8])

long_parts_pl: List[pl.DataFrame] = []

for sh in sheet_names:
    # Load Excel sheet via Polars
    try:
        df_pl = pl.read_excel(source=str(EXCEL_PATH), sheet_name=sh)
    except Exception as e:
        print(f"[warn] skip sheet={sh}: {e!r}")
        continue

    # Normalize column names
    df_pl = df_pl.rename({c: str(c).strip() for c in df_pl.columns})

    # Check required columns (including the optional '노선' safety check)
    if not {"구간", "방향"}.issubset(df_pl.columns):
        print(f"[warn] skipped: required columns missing (sheet={sh})")
        continue

    hour_cols = [c for c in df_pl.columns if c in HOURS]
    if not hour_cols:
        print(f"[warn] skipped: hour columns missing (sheet={sh})")
        continue

    month, day_kind = parse_sheet_name(sh)

    # Include the '노선' column when selecting
    cols_to_select = ["노선", "구간", "방향"] + hour_cols
    if "노선" not in df_pl.columns:
        df_pl = df_pl.with_columns(pl.lit("수도권제1순환선").alias("노선"))

    df_pl = df_pl.select(cols_to_select)
    df_pl = df_pl.with_columns([
        pl.lit(month).alias("월"),
        pl.lit(day_kind).alias("일종"),
    ])

    # Wide -> long, keeping '노선' in the identifier columns
    df_long = df_pl.unpivot(
        index=["노선", "구간", "방향", "월", "일종"],
        on=hour_cols,
        variable_name="시간",
        value_name="지표",
    )
    long_parts_pl.append(df_long)

assert long_parts_pl, "Found no valid sheets to process."

long_pl = pl.concat(long_parts_pl, how="vertical")

# Type cleanup and sort
long_pl = long_pl.with_columns([
    pl.col("시간").cast(pl.Utf8).str.replace("시", "").cast(pl.Int64, strict=False),
    pl.col("지표").cast(pl.Float64, strict=False),
])
long_pl = long_pl.drop_nulls("지표")

print("long_pl shape:", long_pl.shape)
long_df = long_pl
print("long_df shape:", long_df.shape)
display(long_df.head(10))


## 4. Encode IDs and seg_idx
Map weekday/direction to integer IDs and assign `seg_idx` per road segment. Keep `long_df` with columns like [month, weekday_id, direction_id, seg_idx, hour, value].


In [ ]:
from typing import Dict

# 1. Encode weekday and direction IDs (7 weekdays)
DAY2ID: Dict[str, int] = {
    "월요일": 0, "화요일": 1, "수요일": 2, "목요일": 3,
    "금요일": 4, "토요일": 5, "일요일": 6
}
DIR2ID: Dict[str, int] = {"상행": 0, "하행": 1}

# Apply mappings
long_df = long_df.with_columns([
    pl.col("일종").replace(DAY2ID).cast(pl.Int64).alias("요일타입_id"),
    pl.col("방향").replace(DIR2ID).cast(pl.Int64).alias("방향_id"),
])

# 2. Canonicalize section names (treat 'A-B' and 'B-A' as the same physical segment)
def normalize_section_name(name):
    parts = sorted(name.split("-"))
    return "-".join(parts)

# Add normalized name
long_df = long_df.with_columns(
    pl.col("구간").map_elements(normalize_section_name, return_dtype=pl.Utf8).alias("canonical_section")
)

# 3. Create seg_idx from normalized names
unique_sections = (
    long_df
    .select("canonical_section")
    .unique(maintain_order=True)
    .to_series()
    .to_list()
)

SEG2ID: Dict[str, int] = {sec: idx for idx, sec in enumerate(unique_sections)}
S = len(SEG2ID)

# 4. Map seg_idx
long_df = long_df.with_columns(
    pl.col("canonical_section").replace(SEG2ID).cast(pl.Int64).alias("seg_idx")
)

# 5. Sort
long_df = long_df.sort(["월", "요일타입_id", "방향_id", "seg_idx", "시간"])

print(f"Total physical segments S = {S}")
long_df.head(10)


## 5. Build X (group keys) and Y (segment × hour grid)
Compute segment count S and hours T=24. Enumerate group keys (month, weekday_id, direction_id) and build Y tensors per group shaped (S, T).


In [ ]:
# long_df is expected to contain columns (월, 요일타입_id, 방향_id, seg_idx, 시간, 지표)
S = long_df.select(pl.col("seg_idx")).n_unique()
T = 24  # hours 0-23
print("S (segment count) =", S, "T (hour count) =", T)

# Define group axes: (month, weekday_id, direction_id)
group_cols = ["월", "요일타입_id", "방향_id"]

# Assign group indices
group_df = (
    long_df
    .select(group_cols)
    .unique()
    .sort(group_cols)
)

X_keys: list[Tuple[int, int, int]] = [
    (int(row[0]), int(row[1]), int(row[2]))
    for row in group_df.to_numpy()
]

B = len(X_keys)
print("B (group count) =", B)
print("Sample keys (first 5):", X_keys[:5])


In [ ]:
import torch

def build_Y_for_group(df_group: pl.DataFrame, S: int, T: int) -> torch.Tensor:
    """
    df_group: Polars DF containing only rows for the group (월, 요일타입_id, 방향_id)
              columns include seg_idx, 시간, 지표
    """
    # Initialize with zeros
    import numpy as np
    Y_np = np.zeros((S, T), dtype=np.float32)

    for row in df_group.iter_rows(named=True):
        s = int(row["seg_idx"])
        t = int(row["시간"])
        v = float(row["지표"])
        if 0 <= s < S and 0 <= t < T:
            Y_np[s, t] = v

    return torch.from_numpy(Y_np)  # (S, T)


In [ ]:
train_data: Dict[Tuple[int, int, int], torch.Tensor] = {}

for (month, day_id, dir_id) in X_keys:
    # Filter for the group
    df_group = long_df.filter(
        (pl.col("월") == month) &
        (pl.col("요일타입_id") == day_id) &
        (pl.col("방향_id") == dir_id)
    )

    if df_group.height == 0:
        # No data available for this group
        print(f"[warn] No data for group {(month, day_id, dir_id)}.")
        Y_tensor = torch.zeros(S, T, dtype=torch.float32)
    else:
        # Fill Y via NumPy
        Y_np = np.zeros((S, T), dtype=np.float32)
        for row in df_group.iter_rows(named=True):
            s = int(row["seg_idx"])
            t = int(row["시간"])
            v = float(row["지표"])
            if 0 <= s < S and 0 <= t < T:
                Y_np[s, t] = v

        Y_tensor = torch.from_numpy(Y_np)  # (S, T)

    key = (month, day_id, dir_id)
    train_data[key] = Y_tensor

print("len(train_data) =", len(train_data))
some_key = next(iter(train_data.keys()))
print("Example key:", some_key, "Y.shape:", train_data[some_key].shape)


## 6. Configure STNet model
Use Dataset.preprocess to infer feature and label shapes, then set PatchConfig/ModelConfig. The config picks smaller dims on CPU and larger on CUDA.


In [ ]:
from stnet.config import PatchConfig, ModelConfig
from stnet.api import new_model
from stnet.core.system import get_device
from stnet.data.pipeline import Dataset

device = get_device()
print("Device:", device)

ds = Dataset.for_device(device)
feats, labels, keys, label_shape = ds.preprocess(train_data)
# feats: (B, D)   # D = feature dimension derived from the X tuple
# labels: (B, S, T)
print("feats.shape  =", feats.shape)
print("labels.shape =", labels.shape)
print("label_shape  =", label_shape)

B2, D = feats.shape
assert label_shape == (S, T)

patch = PatchConfig(
    is_cube=True,
    grid_size_3d=(S, T, 1),
    patch_size_3d=(1, 1, 1),
    use_padding=True,
)

if device.type == "cuda":
    config = ModelConfig(
        device=device,
        patch=patch,
        normalization_method="layernorm",
        d_model=1024,
        heads=16,
        mlp_ratio=4.0,
        dropout=0.00,
        drop_path=0.00,
        spatial_depth=4,
        temporal_depth=4,
        spatial_latents=64,
        temporal_latents=64,
        modeling_type="spatiotemporal",
        compile_mode="reduce_overhead",
    )
else:
    config = ModelConfig(
        device=device,
        patch=patch,
        normalization_method="layernorm",
        d_model=256,
        heads=2,
        mlp_ratio=2.0,
        dropout=0.05,
        drop_path=0.05,
        spatial_depth=2,
        temporal_depth=2,
        spatial_latents=32,
        temporal_latents=32,
        modeling_type="spatiotemporal",
        compile_mode="disabled",
    )

model = new_model(in_dim=D, out_shape=label_shape, config=config).to(device)


## 7. Train the model
Use `train_data` as a dict mapping `(month, weekday_id, direction_id)` to a `(S, T)` torch tensor with the 24-hour grid. Pass the dict directly into `train`.


In [ ]:
import sys
from stnet.api import train

# Number of samples equals the number of dict entries
num_samples = len(train_data)
base_params = {
    "epochs": 100 if device.type == "cuda" else 50,
    "base_lr": 3e-3,
    "weight_decay": 1e-4,
}

print("num_samples:", num_samples)
print("base_params:", base_params)

# Pass train_data (Dict[X_tuple -> Y_tensor]) directly
model = train(
    model,
    train_data,
    epochs=int(base_params["epochs"]),
    base_lr=float(base_params["base_lr"]),
    weight_decay=float(base_params["weight_decay"]),
    val_frac=0.1,
)


In [ ]:
history = model.history()
for idx, item in enumerate(history):
  print(idx, item)

## 8. Run inference
Build `infer_samples` as a dict keyed by `(month, weekday_id, direction_id)` with `None` values; `predict` fills the `(S, T)` outputs and can return either in-memory or file-backed TensorDicts.


In [ ]:
import os
import torch

from stnet.core.system import get_device
from stnet.api import predict

device = get_device()
print("Device:", device)

infer_samples = {x: None for x, _ in train_data.items()}

# predict: output="memory" -> in-memory TensorDict (X, Y)
results = predict(model, infer_samples, output="memory", path=None)

print(results)
print("results.keys:", list(results.keys()))
print("X shape:", tuple(results["X"].shape))
print("Y shape:", tuple(results["Y"].shape))

# predict: output="file" + path -> PersistentTensorDict (HDF5)
out_path = os.path.abspath("predictions.h5")
results_persistent = predict(model, infer_samples, output="file", path=out_path, overwrite="replace")
print("saved to:", out_path)
print(results_persistent)
print("persistent[0] X shape:", tuple(results_persistent[0]["X"].shape))
print("persistent[0] Y shape:", tuple(results_persistent[0]["Y"].shape))
results_persistent.close()


### Validate TensorDict schema (x/y)
Convert the inference result to a TensorDict with `['x', 'y']` keys to match the expected schema and confirm shapes.


In [ ]:
from tensordict import TensorDict

# Ensure expected keys exist
assert {'X', 'Y'}.issubset(set(results.keys())), 'predict() must return X and Y tensors'
results_xy = TensorDict({'x': results['X'], 'y': results['Y']}, batch_size=results.batch_size)

print('results_xy batch_size:', results_xy.batch_size)
print('x shape:', tuple(results_xy['x'].shape))
print('y shape:', tuple(results_xy['y'].shape))


In [ ]:
# TensorDict allows direct access via td[i]
for i in range(min(5, len(results))):
    td = results[i]
    print("i=", i, "X=", td["X"], "Y.shape=", tuple(td["Y"].shape))


In [ ]:
import numpy as np
import polars as pl

# TensorDict -> map: (month, day_id, dir_id, seg_idx) -> 24-hour prediction vector
X_np = results["X"].detach().cpu().numpy()
Y_np = results["Y"].detach().cpu().numpy()

pred_map = {}
for i, x in enumerate(X_np):
    month, day_id, dir_id = map(int, x.tolist())
    arr = Y_np[i]
    n_segs, n_hours = arr.shape
    for s_idx in range(n_segs):
        pred_map[(month, day_id, dir_id, s_idx)] = arr[s_idx, :24]

rows = []
for (month, day_id, dir_id, s_idx), vec24 in pred_map.items():
    row = {
        'month': month,
        'day_id': day_id,
        'dir_id': dir_id,
        'seg_idx': s_idx,
    }
    for h in range(24):
        row[f'hour_{h}'] = vec24[h]
    rows.append(row)

df_pred = pl.DataFrame(rows)
df_pred = df_pred.sort(['month', 'day_id', 'dir_id', 'seg_idx'])

# Save to Excel via Polars
output_file = 'pred_results_2025.xlsx'
df_pred.write_excel(output_file)
print("Saved:", output_file)
